# 05 · Mejoras y evaluación: baseline frente a final

**Tareas de la práctica que cubre:**

- **Tarea 4:** aplicar las mejoras al sistema y medir su efecto, una a una.
- **Tarea 5:** la evaluación automática con los tres evaluadores (cita, cifra y trayectoria).
- **Entregables:** la tabla *baseline frente a final* (aciertos por familia, recall@5, coste, latencia y llamadas por pregunta, con el mejor valor remarcado) y los ficheros de resultados de cada sistema.

La pregunta que se hará a todos los grupos el día 24 es: **¿las mejoras mejoraron algo de verdad, y a qué coste?** Este notebook la contesta con datos.

## Cómo funciona

1. Se comparan **cuatro sistemas**. El primero es el baseline congelado; cada uno de los otros tres añade **una sola mejora** al anterior. Así se sabe qué hizo cada cosa (medir, arreglar, volver a medir).
2. Cada sistema se ejecuta sobre el **golden propio** (la tabla del informe) y sobre el **oficial** (un control).
3. Los resultados quedan en `resultados/eval_<sistema>_<golden>.csv`. Si el fichero ya existe se reutiliza y no se vuelve a pagar, salvo que se pida repetir.
4. Se construye la tabla con el mejor valor remarcado y una tabla de ablación con lo que aportó cada mejora y lo que costó.
5. Un apartado, **aparte y opcional**, mide la sensibilidad a los **modelos y los embeddings** al estilo del notebook 01. No forma parte de la comparación principal ni cambia el sistema por defecto.
6. Una **tabla de decisiones** reúne todas las opciones exploradas en el proyecto, con su evidencia (leída de los CSV), la decisión tomada y el motivo.

**Coste y tiempo:** unos 2-3 $ y unos 45 minutos en total, con la clave de OpenRouter. El código de cada mejora está en `agente/` (`agente.py`, `guardrails.py`, `retrieval.py`); aquí solo se ejecuta y se mide.

In [ ]:
import os
import sys
from pathlib import Path

RAIZ = Path.cwd() if (Path.cwd() / "agente").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ))

import pandas as pd
from IPython.display import Markdown, display

if not os.environ.get("OPENROUTER_API_KEY"):
    from getpass import getpass
    os.environ["OPENROUTER_API_KEY"] = getpass("OPENROUTER_API_KEY: ")

from agente import agente as ag
from agente import evaluadores, interfaz

RES = RAIZ / "resultados"
GOLDENS = {"propio": RAIZ / "golden/golden_set.jsonl",
           "oficial": RAIZ / "golden/oficial_20.jsonl",
           "huecos": RAIZ / "golden/huecos_humo.jsonl"}
REPETIR = False    # True: vuelve a ejecutar aunque ya exista el CSV (cuesta dinero)

## Los sistemas que se comparan

Cada mejora sale del diagnóstico del notebook 03: se arregla lo que de verdad falló.

| Sistema | Qué añade | Fallos del diagnóstico que ataca |
|---|---|---|
| **baseline** | Búsqueda densa, prompt v1, sin middleware (congelado en el notebook 02) | (punto de partida) |
| **+ prompt v2** | Tres reglas en el prompt: la convención de `cifra`, citas literales y consultas con el vocabulario del informe | 6 fallos de "variación en `cifra`" y 2 de cita |
| **+ guardrails** | Límites de llamadas, reintento con espera y verificador de cifras y de formato (notebook 04) | Cifras sin respaldo, unidades, respuestas sin esquema |
| **+ híbrida (final)** | `search_filings` pasa a búsqueda híbrida (BM25 + densa, fusión RRF) | 4 fallos de retrieval |

Un aviso para leer los resultados: la búsqueda híbrida mejora sobre todo la métrica de **recall**. De los 4 fallos de retrieval del baseline, solo uno acabó en una respuesta mala; los otros 3 se contestaron bien pese a todo. Y los guardrails **cuestan** llamadas extra: cada corrección es una vuelta más del modelo.

Estas son las configuraciones tal como están en el código, y lo que añade el prompt v2:

In [ ]:
configuraciones = pd.DataFrame(ag.CONFIGURACIONES).T
configuraciones["prompt"] = configuraciones["prompt"].map(lambda p: "v2" if p is ag.SYSTEM_V2 else "v1")
display(configuraciones)

print("Lo que añade el prompt v2 al v1:")
print(ag.SYSTEM_V2[len(ag.SYSTEM):])

## El baseline: no se vuelve a ejecutar

El baseline se congeló en el notebook 02 sobre el golden propio final, y **sus CSV no se tocan**: son la mitad de la tabla. Aquí solo se leen. Si hiciera falta regenerarlo, el mismo código lo produce con `config="baseline"` (prompt v1, búsqueda densa, sin middleware), pero eso no forma parte de esta comparación.

**Cómo se cuentan los errores.** Una pregunta que termina en error (por ejemplo, un modelo que no devuelve la respuesta estructurada) **cuenta como fallo** en todas las métricas que le aplican, no como una pregunta ausente. Si no, un sistema que pierde preguntas parecería mejor: acertaría el 100 % de las que sí contesta. La función `ajustar` hace ese recuento en memoria; los CSV originales no se modifican.

In [ ]:
# Qué métricas se aplican a cada familia de pregunta
APLICA = {"cifra_ok": ("numerica", "comparativa"),
          "cita_ok": ("extractiva", "comparativa"),
          "recall": ("extractiva", "comparativa"),
          "tool_ok": ("numerica", "extractiva", "comparativa")}


def ajustar(df):
    """Copia de la tabla en la que una pregunta con error cuenta como fallo, no como ausente."""
    df = df.copy()
    con_error = df["error"].notna()
    for columna, familias in APLICA.items():
        df[columna] = df[columna].astype(object)
        df.loc[con_error & df["familia"].isin(familias), columna] = False
    return df


EVAL = {("baseline", nombre): ajustar(pd.read_csv(RES / f"eval_baseline_{nombre}.csv"))
        for nombre in ("propio", "oficial", "huecos")}
print({clave: len(df) for clave, df in EVAL.items()}, "filas del baseline cargadas")

## Ejecutar los sistemas

La función `correr(sistema, golden)` hace tres cosas:

1. Llama a `interfaz.evaluar(...)`, la misma función que se ejecutará el día 24 sobre el hold-out. Cada pregunta corre en su propio hilo, y se aplican los tres evaluadores.
2. La columna `recall` se mide con **la búsqueda de ese sistema** (densa o híbrida), no con la final para todos, para que la comparación sea justa.
3. Guarda el resultado en `resultados/eval_<sistema>_<golden>.csv`. Si ya existe, lo reutiliza.

Además de las métricas, cada fila guarda **qué respondió el agente** (`cifra_agente`, `fuente`, las herramientas usadas y el número de avisos de los guardrails), para poder diagnosticar un fallo sin repetir la pregunta.

Por seguridad, `correr` se niega a ejecutar el baseline: así no hay forma de sobrescribir los CSV congelados.

In [ ]:
def correr(sistema, golden):
    """Evalúa un sistema sobre un golden y devuelve su tabla. Reutiliza el CSV si ya existe."""
    if sistema == "baseline":
        raise ValueError("El baseline está congelado: no se vuelve a ejecutar aquí.")
    ruta = RES / f"eval_{sistema}_{golden}.csv"
    if ruta.exists() and not REPETIR:
        print(f"{sistema} · {golden}: ya existe, se reutiliza")
        return ajustar(pd.read_csv(ruta))
    print(f"{sistema} · {golden}: ejecutando…")
    return ajustar(interfaz.evaluar(str(GOLDENS[golden]), etiqueta=f"{sistema}_{golden}", config=sistema))

## Sistema 1 — Baseline + prompt v2

Solo cambia el prompt. Es la mejora más barata y la que, según el diagnóstico, ataca más fallos: la mitad eran el mismo error de convención (el agente escribía la variación en `cifra` en lugar del valor del ejercicio).

**Qué mirar en el resultado:** la columna `cifra` de las comparativas y la de `cita`. Si el prompt basta, aquí ya se ve casi toda la mejora, y sin coste extra: no añade llamadas.

Coste aproximado: 60 ¢ (unos 14 minutos).

In [ ]:
EVAL["prompt_v2", "propio"] = correr("prompt_v2", "propio")
EVAL["prompt_v2", "oficial"] = correr("prompt_v2", "oficial")

## Sistema 2 — Más los guardrails

Se añaden los cuatro middleware del notebook 04: límite de llamadas a herramienta, límite de llamadas al modelo, reintento con espera y el verificador de cifras (que también exige la salida estructurada).

**Qué mirar en el resultado:**

- La columna `avisos` de la ablación: cuántas veces tuvo que intervenir el verificador. Si el prompt v2 ya arregló las cifras, deberían ser pocos.
- El **coste y la latencia**: cada aviso es una llamada más, así que esta mejora se paga.
- Que no aparezcan errores: el reintento existe para eso.

In [ ]:
EVAL["guardrails", "propio"] = correr("guardrails", "propio")
EVAL["guardrails", "oficial"] = correr("guardrails", "oficial")

## Sistema 3 — Más la búsqueda híbrida (el sistema final)

La herramienta `search_filings` pasa de búsqueda densa a híbrida. Es la única mejora que toca el retrieval, y por eso es la única que mueve la columna `recall@5`.

También se ejecutan aquí los **5 huecos** (preguntas sin respuesta en el corpus). No son para la tabla principal, sino para comprobar que los guardrails y el nuevo prompt **no estropean** algo que en el baseline ya funcionaba: responder `fuente='ninguna'` en lugar de inventarse una cifra. Al menos 2 de las 10 preguntas ciegas serán así.

In [ ]:
EVAL["final", "propio"] = correr("final", "propio")
EVAL["final", "oficial"] = correr("final", "oficial")
EVAL["final", "huecos"] = correr("final", "huecos")

## La tabla del informe

Una fila por sistema, con las columnas que pide el enunciado:

| Columna | Qué es | Mejor |
|---|---|---|
| `cita` | Proporción de preguntas con la cita correcta | más alto |
| `cifra` | Proporción con la cifra igual a XBRL (tolerancia del 1 %) | más alto |
| `trayectoria` | Proporción que usó la herramienta esperada | más alto |
| `recall@5` | Proporción de anclas que entran en el top-5 de la búsqueda | más alto |
| `coste (¢)` | Coste medio por pregunta, en céntimos | más bajo |
| `latencia (s)` | Segundos medios por pregunta | más bajo |
| `llamadas` | Llamadas a herramienta por pregunta | más bajo |
| `errores` | Preguntas que terminaron en error | más bajo |

El mejor valor de cada columna va **en negrita**. La tabla se guarda como CSV y como Markdown, lista para pegar en el informe.

**Cómo leerla.** Con 13 preguntas evaluables por golden, una pregunta vale unos 7,7 puntos. Una diferencia de una sola pregunta es ruido; el modelo no es determinista y la misma pregunta puede acertar o fallar de una tirada a otra. Por eso la siguiente tabla cuenta preguntas, no porcentajes. Y una pregunta que terminó en error cuenta como fallo, así que la columna `errores` explica por qué a veces bajan `cita` o `cifra`. La latencia media es sensible a una sola pregunta atascada; bajo la tabla se listan las atípicas.

In [ ]:
NOMBRES = {"baseline": "baseline", "prompt_v2": "+ prompt v2",
           "guardrails": "+ guardrails", "final": "+ híbrida (final)"}
COLUMNAS = {"cita_ok": "cita", "cifra_ok": "cifra", "tool_ok": "trayectoria", "recall@5": "recall@5",
            "coste_medio_¢": "coste (¢)", "latencia_media_s": "latencia (s)",
            "tools_por_pregunta": "llamadas", "errores": "errores"}
MAS_ES_MEJOR = ["cita", "cifra", "trayectoria", "recall@5"]


def tabla(golden):
    """Una fila por sistema, con las métricas de la tabla del informe."""
    filas = [evaluadores.resumir(EVAL[sistema, golden], NOMBRES[sistema]) for sistema in NOMBRES]
    return pd.DataFrame(filas).set_index("sistema")[list(COLUMNAS)].rename(columns=COLUMNAS)


def a_markdown(t):
    """La tabla en Markdown, con el mejor valor de cada columna en negrita."""
    lineas = ["| sistema | " + " | ".join(t.columns) + " |", "|---" * (len(t.columns) + 1) + "|"]
    for nombre, fila in t.iterrows():
        celdas = []
        for columna in t.columns:
            valor = fila[columna]
            texto = "–" if pd.isna(valor) else f"{valor:.2f}"
            mejor = t[columna].max() if columna in MAS_ES_MEJOR else t[columna].min()
            if pd.notna(valor) and valor == mejor and t[columna].nunique() > 1:
                texto = f"**{texto}**"
            celdas.append(texto)
        lineas.append(f"| {nombre} | " + " | ".join(celdas) + " |")
    return "\n".join(lineas)


t_propio = tabla("propio")
texto = a_markdown(t_propio)
display(Markdown("### Golden propio (la tabla del informe)\n\n" + texto))

t_propio.round(3).to_csv(RES / "tabla_baseline_vs_final_propio.csv")
(RES / "tabla_baseline_vs_final_propio.md").write_text(texto, encoding="utf-8")
print("Guardado: resultados/tabla_baseline_vs_final_propio.{csv,md}")

print("\nAciertos por familia, baseline frente a final:")
por_familia = pd.concat({NOMBRES[s]: evaluadores.por_familia(EVAL[s, "propio"]) for s in ("baseline", "final")}, axis=1)
display(por_familia.round(2))

# La latencia media es sensible a una sola pregunta atascada: se avisa de las atípicas
print("""Latencias atípicas (más de 5 veces la mediana de su sistema):""")
hay = False
for golden in ("propio", "oficial"):
    for sistema in NOMBRES:
        df = EVAL[sistema, golden]
        for r in df[df["latencia_s"] > 5 * df["latencia_s"].median()].itertuples():
            hay = True
            print(f"  {NOMBRES[sistema]} · {golden} · {r.id}: {r.latencia_s:.0f} s (mediana {df['latencia_s'].median():.0f} s)")
if not hay:
    print("  ninguna")

## Qué mejoró y a qué coste: la ablación

La tabla anterior compara los extremos. Esta cuenta, en cada sistema, las **preguntas acertadas** y **cuántos fallos se evitan** al añadir cada mejora, que es lo que permite atribuir cada efecto a su causa:

- `cifra`, `cita` y `recall@5`, como *acertadas / evaluables*. Las preguntas que terminaron en error ya cuentan como fallo.
- `errores`: preguntas que el sistema no llegó a contestar.
- `avisos`: cuántas veces intervinieron los guardrails en todo el golden.
- El coste y la latencia, por si una mejora se paga cara.

Se comparan **fallos evitados** y no aciertos, porque el número de preguntas evaluables cambia de un sistema a otro (por ejemplo, un agente que empieza a citar en las numéricas añade citas evaluables). La última celda de esta sección aplica una regla fijada de antemano: una mejora cuenta como **clara** si evita al menos **2 fallos** en alguna métrica, y como **negativa** si causa 2 o más. Con menos, está dentro del ruido y se dice como tal.

In [ ]:
def n_ok(df, columna):
    """(preguntas acertadas, preguntas evaluables) de una columna."""
    validas = df[columna].dropna()
    return int((validas == True).sum()), len(validas)   # noqa: E712


def fallos(df, columna):
    """Preguntas fallidas de una columna. Las que terminaron en error ya cuentan como fallo."""
    ok, n = n_ok(df, columna)
    return n - ok


filas = []
for sistema in NOMBRES:
    df = EVAL[sistema, "propio"]
    fila = {"sistema": NOMBRES[sistema]}
    for columna, nombre in (("cifra_ok", "cifra"), ("cita_ok", "cita"), ("recall", "recall@5")):
        ok, n = n_ok(df, columna)
        fila[nombre] = f"{ok}/{n}"
    fila["errores"] = int(df["error"].notna().sum())
    fila["avisos"] = int(df["n_avisos"].fillna(0).sum()) if "n_avisos" in df else "–"
    fila["coste (¢)"] = round(df["coste_usd"].mean() * 100, 2)
    fila["latencia (s)"] = round(df["latencia_s"].mean(), 1)
    filas.append(fila)
display(pd.DataFrame(filas).set_index("sistema"))

print("Fallos que se evitan al añadir cada mejora (golden propio):\n")
anterior = None
for sistema in NOMBRES:
    df = EVAL[sistema, "propio"]
    if anterior is not None:
        evitados = {nombre: fallos(anterior, col) - fallos(df, col)
                    for col, nombre in (("cifra_ok", "cifra"), ("cita_ok", "cita"), ("recall", "recall@5"))}
        d_coste = (df["coste_usd"].mean() - anterior["coste_usd"].mean()) * 100
        d_lat = df["latencia_s"].mean() - anterior["latencia_s"].mean()
        if min(evitados.values()) <= -2:
            veredicto = "empeora"
        elif max(evitados.values()) >= 2:
            veredicto = "mejora clara"
        else:
            veredicto = "dentro del ruido"
        print(f"{NOMBRES[sistema]:<20} " + " · ".join(f"{k} {v:+d}" for k, v in evitados.items())
              + f" fallos evitados · coste {d_coste:+.2f} ¢ · latencia {d_lat:+.1f} s   → {veredicto}")
    anterior = df

## El golden oficial, como control

Es la misma tabla sobre las 20 preguntas oficiales. **No es la tabla del informe**, y hay que leerla con una reserva: sus anclas se usaron para medir el retrieval en el notebook 01, así que tampoco es un conjunto completamente virgen. Sirve para comprobar que lo que mejora en el golden propio no es una casualidad de esas 20 preguntas.

Debajo, los **huecos**: baseline frente a sistema final. La columna que importa es `cifra`, que aquí vale 1 solo si el agente respondió sin cifra y con `fuente='ninguna'`.

In [ ]:
t_oficial = tabla("oficial")
texto_oficial = a_markdown(t_oficial)
display(Markdown("### Golden oficial (control)\n\n" + texto_oficial))
t_oficial.round(3).to_csv(RES / "tabla_baseline_vs_final_oficial.csv")

huecos = pd.DataFrame([evaluadores.resumir(EVAL[s, "huecos"], NOMBRES[s]) for s in ("baseline", "final")])
huecos = huecos.set_index("sistema")[["cifra_ok", "coste_medio_¢", "latencia_media_s", "errores"]]
display(Markdown("### Huecos (sin respuesta en el corpus)"))
display(huecos.round(3))

## Ruido y regenerabilidad (opcional)

Esta celda vuelve a ejecutar el **baseline con el mismo código** (`config="baseline"`) y lo compara con el congelado. Sirve para dos cosas:

1. **Medir el ruido.** Si el mismo sistema, con el mismo código, cambia de una tirada a otra en 1 o 2 preguntas, esa es la diferencia mínima que hay que superar para decir que una mejora es real.
2. **Demostrar que el baseline es regenerable** ejecutando el repositorio, como pide el enunciado.

Escribe en `resultados/eval_baseline_repeticion_propio.csv`, **nunca** sobre los CSV congelados. Cuesta unos 30 ¢; se activa cambiando `MEDIR_RUIDO` a `True`.

In [ ]:
MEDIR_RUIDO = False

if MEDIR_RUIDO:
    repeticion = interfaz.evaluar(str(GOLDENS["propio"]), etiqueta="baseline_repeticion_propio", config="baseline")
    print("\nBaseline congelado frente a su repetición (golden propio):")
    for columna in ("cifra_ok", "cita_ok", "recall"):
        print(f"  {columna:<9} congelado {n_ok(EVAL['baseline', 'propio'], columna)}   "
              f"repetición {n_ok(repeticion, columna)}")
else:
    print("Desactivado (MEDIR_RUIDO = False).")

## Sensibilidad a modelos y embeddings (al estilo del notebook 01)

Los cuatro sistemas de arriba mantienen **fijos** el modelo del agente (`gemini-3.8-flash`) y el embedding (`bge-small`), para que cada diferencia se deba a una sola mejora. Pero cambiar de modelo o de embedding también es una palanca, y merece medirse. Se hace **aparte**, como en el notebook 01: en su propio apartado, con sus propios ficheros de resultados y **sin tocar las configuraciones** ni el sistema por defecto.

Hay dos preguntas distintas, con costes muy distintos:

| Pregunta | Cómo se mide | Coste |
|---|---|---|
| ¿Otro **embedding** o otro LLM de **reescritura** recuperan mejor? | A nivel de retrieval, sin agente: es la matriz del notebook 01 | Ya está hecho |
| ¿Otro **modelo del agente** responde mejor o más barato? | De punta a punta, con el sistema final: hay que ejecutar el agente | Sí (opcional) |

### Parte A — Embeddings y reescritura (matriz del notebook 01)

Se lee la matriz que dejó el notebook 01: 5 embeddings, varios LLM de reescritura y los dos modos de búsqueda, sobre las 26 anclas de los dos golden. Lo que interesa es cuánto se separa la mejor combinación del **control**, que es lo que usa hoy el sistema: búsqueda híbrida con `bge-small` local (el LLM de reescritura, `gemini-3.5-flash-lite`, es solo el instrumento de medida).

In [ ]:
ruta_matriz = RES / "matriz_modelos_recall.csv"

if ruta_matriz.exists():
    matriz = pd.read_csv(ruta_matriz)
    con_reescritura = matriz[matriz.llm != "ES sin reescribir"]
    n = int(matriz.n.iloc[0])
    control = con_reescritura[(con_reescritura.modo == "hibrida")
                              & (con_reescritura.llm == "gemini-3.5-flash-lite (ctl)")
                              & (con_reescritura.embedding == "bge-small (ctl, local)")].aciertos.iloc[0]

    print(f"Anclas recuperadas en el top-5, de {n} (consulta reescrita en inglés):")
    display(con_reescritura.pivot_table(index="embedding", columns=["modo", "llm"], values="aciertos"))

    mejor = con_reescritura.loc[con_reescritura.aciertos.idxmax()]
    print(f"Control (híbrida, bge-small, gemini-3.5-flash-lite): {int(control)}/{n}")
    print(f"Mejor de las {len(con_reescritura)} combinaciones: {int(mejor.aciertos)}/{n} "
          f"({mejor.modo}, {mejor.embedding}, {mejor.llm})")
    empatadas = int((con_reescritura.aciertos == mejor.aciertos).sum())
    print(f"Combinaciones que alcanzan ese máximo: {empatadas} (se muestra la primera)")
    print(f"Ventaja del mejor sobre el control: {int(mejor.aciertos) - int(control):+d} ancla(s)  "
          f"(una ancla = ±{1 / n:.3f})")
else:
    print("No hay matriz: ejecuta antes el notebook 01, que la genera.")

**Cómo leer la parte A.** Es un máximo entre muchas combinaciones sobre pocas anclas, así que sobreestima: el mejor de tantos resultados casi siempre tiene algo de suerte. Si la ventaja sobre el control es de una o dos anclas, es ruido y no justifica cambiar de embedding, con el coste añadido de una llamada a la API por consulta y un índice nuevo que el clon limpio tendría que regenerar. Esta comprobación es la que respalda dejar el embedding fijo en la comparación principal.

### Parte B — El modelo del agente (opcional, de punta a punta)

Aquí no hay atajo: el modelo del agente decide qué herramientas llama y qué escribe en `cifra` y en `cita`, así que solo se puede medir ejecutándolo. Se prueba **el sistema final con otro LLM**, sobre el golden propio, y se compara con el mismo sistema final con el modelo actual. Cada modelo escribe en `resultados/eval_modelo_<nombre>_propio.csv`.

Cuatro reservas:

- **El coste depende del precio de cada modelo.** Se registran abajo, tomados del catálogo público de OpenRouter, porque sin precio la columna de coste saldría a 0.
- **Un modelo puede no admitir la salida estructurada nativa** a través de OpenRouter. Si es así, sus filas aparecerán como error, y eso también es un resultado.
- **Una sola tirada por modelo.** Con el ruido de 1 o 2 preguntas, solo una diferencia clara cuenta.
- **No se compara con el baseline**, que se congeló con `gemini-3.8-flash`, sino con el sistema final con ese mismo modelo.

Se activa cambiando `PROBAR_MODELOS` a `True`. Coste: unos céntimos por modelo.

In [ ]:
from agente import trazas

PROBAR_MODELOS = False    # True: ejecuta los modelos alternativos (cuesta dinero)

MODELOS = {   # nombre: (id en OpenRouter, $/M tokens de entrada, $/M de salida)
    "gemini-3.5-flash-lite": ("openrouter:google/gemini-3.5-flash-lite", 0.30, 2.50),
    "deepseek-v4-flash": ("openrouter:deepseek/deepseek-v4-flash", 0.04, 0.08),
}
for nombre, (modelo, precio_entrada, precio_salida) in MODELOS.items():
    trazas.PRECIOS_OPENROUTER[modelo.split(":", 1)[-1]] = (precio_entrada, precio_salida)

EVAL_MODELOS = {}
if PROBAR_MODELOS:
    for nombre, (modelo, _, _) in MODELOS.items():
        ruta = RES / f"eval_modelo_{nombre}_propio.csv"
        if ruta.exists() and not REPETIR:
            EVAL_MODELOS[nombre] = ajustar(pd.read_csv(ruta))
        else:
            EVAL_MODELOS[nombre] = ajustar(interfaz.evaluar(str(GOLDENS["propio"]), etiqueta=f"modelo_{nombre}_propio",
                                                            config="final", modelo=modelo))

    filas = [evaluadores.resumir(EVAL["final", "propio"], "gemini-3.8-flash (actual)")]
    filas += [evaluadores.resumir(df, nombre) for nombre, df in EVAL_MODELOS.items()]
    display(pd.DataFrame(filas).set_index("sistema")[list(COLUMNAS)].rename(columns=COLUMNAS).round(2))
else:
    print("Desactivado (PROBAR_MODELOS = False).")

## La tabla de decisiones

Las opciones se han ido explorando en cuatro notebooks: la escalera y la matriz de retrieval (01), el diagnóstico (03), los guardrails (04) y las mejoras del sistema (aquí). Esta tabla las reúne **todas en un solo sitio**, con tres cosas por opción:

- **Evidencia:** lo que se midió. No está escrita a mano: se lee de los CSV de `resultados/`.
- **Decisión:** qué se hizo con esa opción.
- **Motivo:** por qué.

| Decisión | Significa |
|---|---|
| ✔ Adoptada | Forma parte del sistema final |
| ✖ Descartada | Se probó y no se usa, porque no mejoró o empeoró |
| ◐ Se mantiene | No mejoró el golden de forma clara, pero se conserva por diseño (robustez o coherencia con otra decisión) |
| ◐ Candidata | Podría mejorar algo, pero no cumple el criterio para adoptarla; se confirma en el hold-out |
| ○ Referencia | Punto de partida o instrumento de medida, no una alternativa |
| ○ Sin evaluar | No hay datos para decidir |

Las reglas que gobiernan las decisiones son pocas y se aplican igual en todas partes:

1. **Una mejora cuenta como clara si evita al menos 2 fallos** (preguntas o anclas). Con menos, es ruido: una sola vale ±0,04-0,08 y el modelo no es determinista. Las preguntas que terminan en error cuentan como fallo.
2. **Solo se cambia una cosa a la vez**, para poder atribuir cada efecto.
3. **No se adopta lo que añade dependencias por una ventaja de ruido:** un embedding de pago exige una llamada a la API por consulta y un índice que el clon limpio tendría que regenerar.

La tabla se guarda en `resultados/tabla_decisiones.{csv,md}`, lista para el informe. Si falta algún CSV (por ejemplo, no se ejecutó la parte B), esa fila indica que no hay datos.

In [ ]:
def leer(nombre):
    """Un CSV de resultados/, o None si no existe."""
    ruta = RES / nombre
    return pd.read_csv(ruta) if ruta.exists() else None


escalera = {golden: leer(f"recall_escalera_{golden}.csv") for golden in ("propio", "oficial")}
matriz = leer("matriz_modelos_recall.csv")
diagnostico = leer("diagnostico_baseline.csv")

ADOPTADA, DESCARTADA = "✔ Adoptada", "✖ Descartada"
MANTENIDA, CANDIDATA = "◐ Se mantiene", "◐ Candidata"
REFERENCIA, SIN_EVALUAR = "○ Referencia", "○ Sin evaluar"

FILAS = []


def opcion(area, nombre, evidencia, decision, motivo):
    """Añade una fila a la tabla de decisiones."""
    FILAS.append({"área": area, "opción": nombre, "evidencia (CSV)": evidencia,
                  "decisión": decision, "motivo": motivo})


# ---- Retrieval 1: la escalera del notebook 01 ----------------------------
if all(f is not None for f in escalera.values()):
    n_anclas = int(escalera["propio"]["n_ancladas"].iloc[0])

    def anclas(letra):
        """Anclas recuperadas en el top-5 por un escalón, en cada golden."""
        salida = {}
        for golden, f in escalera.items():
            linea = f[f["escalón"] == letra].iloc[0]
            salida[golden] = round(linea["recall@5"] * linea["n_ancladas"])
        return salida

    def texto(letra):
        a = anclas(letra)
        return f"propio {a['propio']}/{n_anclas} · oficial {a['oficial']}/{n_anclas}"

    total = {letra: sum(anclas(letra).values()) for letra in "ABCD"}
    area = "Retrieval · escalera"

    opcion(area, "A · Densa plana (pregunta en español, sin filtros)", texto("A"), REFERENCIA,
           "Punto de partida del día 10: mezcla compañías y ejercicios")
    d = total["B"] - total["A"]
    opcion(area, "B · + filtros de metadatos", texto("B"), ADOPTADA if d >= 2 else DESCARTADA,
           f"{d:+d} ancla(s) sobre A: busca solo en el documento correcto. El agente pasa los filtros a search_filings")
    d = total["C"] - total["B"]
    opcion(area, "C · + búsqueda híbrida (BM25 + densa), consulta aún en español", texto("C"),
           ADOPTADA if d >= 2 else DESCARTADA,
           f"{d:+d} ancla(s) sobre B: con la consulta en español, BM25 casi no encuentra palabras en común. Por sí sola no sirve")
    d = total["D"] - total["C"]
    opcion(area, "D · + reescritura de la consulta ES→EN", texto("D"), ADOPTADA if d >= 2 else DESCARTADA,
           f"{d:+d} ancla(s) sobre C: el cuello era el idioma. Se consigue pidiendo al agente que escriba en inglés (prompt), sin un paso extra dentro de la tool")

# ---- Retrieval 2: la matriz del notebook 01 -------------------------------
if matriz is not None:
    n = int(matriz.n.iloc[0])
    en_ingles = matriz[matriz.llm != "ES sin reescribir"]
    es_control = ((en_ingles.modo == "hibrida") & (en_ingles.llm == "gemini-3.5-flash-lite (ctl)")
                  & (en_ingles.embedding == "bge-small (ctl, local)"))
    control = int(en_ingles[es_control].aciertos.iloc[0])
    pares = en_ingles.pivot_table(index=["embedding", "llm"], columns="modo", values="aciertos")
    area = "Retrieval · matriz"

    local = pares.loc["bge-small (ctl, local)"]
    opcion(area, "Búsqueda híbrida frente a densa (consulta en inglés)",
           f"híbrida ≥ densa en {int((pares.hibrida >= pares.densa).sum())}/{len(pares)} parejas (embedding, LLM), "
           f"mejor en {int((pares.hibrida > pares.densa).sum())} · con bge-small gana {(local.hibrida - local.densa).mean():+.1f} anclas de media",
           ADOPTADA, "Nunca pierde frente a la densa y compensa un embedding débil como el local")

    es = matriz[matriz.llm == "ES sin reescribir"].pivot_table(index="embedding", columns="modo", values="aciertos")
    opcion(area, "Consulta en español, sin reescribir",
           f"la híbrida empeora a la densa en {int((es.hibrida < es.densa).sum())} de {len(es)} embeddings",
           DESCARTADA, "BM25 necesita solape léxico: hay que consultar en inglés")

    for embedding, sub in en_ingles.groupby("embedding", sort=False):
        hibrida = sub[sub.modo == "hibrida"].aciertos
        mejor, peor = int(hibrida.max()), int(hibrida.min())
        es_local = "local" in embedding
        d = mejor - control
        evidencia = f"híbrida {peor}–{mejor}/{n} según el LLM · {'local' if es_local else 'por API'}"
        if es_local:
            decision, motivo = ADOPTADA, f"Es el control: sin dependencias ni coste, y con la híbrida llega a {control}/{n}"
        elif d >= 1:
            decision, motivo = CANDIDATA, (f"{d:+d} ancla(s) sobre el control: no llega a 2, dentro del ruido. "
                                           f"Exige API e índice nuevo; se confirma en el hold-out")
        else:
            decision, motivo = DESCARTADA, f"No supera al control ({control}/{n}) y exige API"
        opcion(area, f"Embedding {embedding.replace(' (ctl, local)', '')}", evidencia, decision, motivo)

    for llm, sub in en_ingles.groupby("llm", sort=False):
        con_local = sub[sub.embedding == "bge-small (ctl, local)"].set_index("modo").aciertos
        evidencia = f"con bge-small: densa {int(con_local['densa'])}/{n} · híbrida {int(con_local['hibrida'])}/{n}"
        if "ctl" in llm:
            decision, motivo = REFERENCIA, "Instrumento de medida (columna recall y matriz); no forma parte de la tool"
        elif llm == "gemini-3.8-flash":
            decision, motivo = REFERENCIA, "Es el modelo del propio agente: la reescritura la hace él mismo al escribir su consulta"
        elif int(con_local["hibrida"]) > control:
            decision, motivo = CANDIDATA, f"Supera al control ({control}/{n}) con el embedding local"
        else:
            decision, motivo = DESCARTADA, f"No supera al control ({control}/{n}) con el embedding local"
        opcion(area, f"LLM de reescritura {llm.replace(' (ctl)', '')}", evidencia, decision, motivo)

In [ ]:
# ---- Sistema: las mejoras medidas en este notebook -------------------------
def preguntas_con(prefijos):
    """Preguntas del diagnóstico del baseline cuya causa empieza por alguno de los prefijos."""
    if diagnostico is None:
        return "?"
    return diagnostico[diagnostico.causa.str.startswith(tuple(prefijos))]["id"].nunique()


def linea(df):
    """Resumen de un sistema: cifra, cita y recall como aciertos/evaluables, errores, coste y latencia."""
    partes = [f"{nombre} {n_ok(df, col)[0]}/{n_ok(df, col)[1]}"
              for col, nombre in (("cifra_ok", "cifra"), ("cita_ok", "cita"), ("recall", "recall"))]
    errores_df = int(df["error"].notna().sum())
    if errores_df:
        partes.append(f"{errores_df} errores")
    return " · ".join(partes) + f" · {df['coste_usd'].mean() * 100:.2f} ¢ · {df['latencia_s'].mean():.0f} s"


def cambio(anterior, sistema, golden="propio"):
    """Fallos que se evitan al pasar de un sistema al siguiente (negativo = se causan)."""
    return {nombre: fallos(EVAL[anterior, golden], col) - fallos(EVAL[sistema, golden], col)
            for col, nombre in (("cifra_ok", "cifra"), ("cita_ok", "cita"), ("recall", "recall"))}


def errores(sistema):
    """Preguntas sin respuesta de un sistema, sumando el golden propio y el oficial."""
    return sum(int(EVAL[sistema, g]["error"].notna().sum()) for g in ("propio", "oficial"))


ORDEN = ["baseline", "prompt_v2", "guardrails", "final"]
area = "Sistema · mejoras"
opcion(area, "Baseline (densa, prompt v1, sin middleware)", linea(EVAL["baseline", "propio"]), REFERENCIA,
       "Punto de partida congelado en el notebook 02")

MEJORAS = {   # sistema: (nombre, qué falla del diagnóstico ataca, decisión si no es clara, motivo)
    "prompt_v2": ("+ Prompt v2 (convención de cifra, citas literales, vocabulario del informe)",
                  f"ataca {preguntas_con(['cifra: variación', 'cifra: valor'])} fallos de cifra y {preguntas_con(['cita'])} de cita",
                  MANTENIDA, "Es la base sobre la que actúan los guardrails"),
    "guardrails": ("+ Guardrails (límites, reintento, verificador de cifras y de formato)",
                   f"ataca {preguntas_con(['cifra: sin respaldo'])} cifra sin respaldo, unidades y respuestas sin esquema",
                   MANTENIDA, "Se conserva como red de seguridad"),
    "final": ("+ Búsqueda híbrida en search_filings (= sistema final)",
              f"ataca {preguntas_con(['retrieval'])} fallos de retrieval",
              MANTENIDA, "Sigue la decisión del notebook 01: no pierde frente a la densa y compensa el embedding local"),
}
for anterior, sistema in zip(ORDEN, ORDEN[1:]):
    nombre, ataca, si_no_es_clara, motivo_no_clara = MEJORAS[sistema]
    c = cambio(anterior, sistema)
    evitados = ", ".join(f"{k} {v:+d}" for k, v in c.items())
    e0, e1 = errores(anterior), errores(sistema)
    nota_errores = f" Preguntas sin respuesta: {e0} → {e1}." if e0 != e1 else ""
    if min(c.values()) <= -2:
        decision, motivo = DESCARTADA, f"Empeora (fallos evitados: {evitados}).{nota_errores}"
    elif max(c.values()) >= 2:
        decision, motivo = ADOPTADA, f"Mejora clara (fallos evitados: {evitados}).{nota_errores}"
    else:
        decision, motivo = si_no_es_clara, f"Dentro del ruido (fallos evitados: {evitados}).{nota_errores} {motivo_no_clara}"
    opcion(area, nombre, f"{ataca} · {linea(EVAL[sistema, 'propio'])}", decision, motivo)

# ---- Sistema: los guardrails, con la evidencia de los CSV --------------------
area = "Sistema · guardrails"
maximo = int(max(EVAL[s, g]["n_tools"].max() for s in ("guardrails", "final") for g in ("propio", "oficial")))
opcion(area, "Límites de llamadas (12 a herramientas, 16 al modelo) y reintento con espera",
       f"máx. {maximo} llamadas a herramienta en una pregunta (límite 12)",
       ADOPTADA if maximo >= 12 else MANTENIDA,
       "El límite actuó" if maximo >= 12 else "No llegó a activarse en estos golden; protege frente a bucles y a los límites de peticiones (el día 24 hay 20 por minuto)")

avisos = sum(int(EVAL["guardrails", g]["n_avisos"].fillna(0).sum()) for g in ("propio", "oficial") if "n_avisos" in EVAL["guardrails", g])
e_antes, e_despues = errores("prompt_v2"), errores("guardrails")
opcion(area, "Verificador de cifras (tolerancia 0,1 %) y salida estructurada obligatoria",
       f"{avisos} avisos en las 40 preguntas del propio y el oficial · preguntas sin respuesta: prompt v2 {e_antes} → guardrails {e_despues}",
       ADOPTADA if (avisos > 0 or e_despues < e_antes) else MANTENIDA,
       "Recupera las preguntas que quedaban sin respuesta estructurada y corrige cifras" if e_despues < e_antes
       else "Corrige cifras" if avisos > 0 else "No hizo falta intervenir; se conserva como red de seguridad")

# ---- Modelo del agente: parte B ---------------------------------------------
def fallos_totales(df):
    """Fallos de cifra, cita y trayectoria juntos (menos es mejor)."""
    return sum(fallos(df, col) for col in ("cifra_ok", "cita_ok", "tool_ok"))


area = "Modelo del agente"
actual = EVAL["final", "propio"]
opcion(area, "gemini-3.8-flash (actual)", linea(actual), ADOPTADA,
       "Es el modelo del baseline: cambiarlo mezclaría dos efectos")
alternativos = {ruta.stem[len("eval_modelo_"):-len("_propio")]: ajustar(pd.read_csv(ruta))
                for ruta in sorted(RES.glob("eval_modelo_*_propio.csv"))}
if alternativos:
    for nombre, df in alternativos.items():
        d_fallos = fallos_totales(df) - fallos_totales(actual)
        coste = df["coste_usd"].mean() / actual["coste_usd"].mean()
        latencia = df["latencia_s"].mean() / actual["latencia_s"].mean()
        detalle = f"{d_fallos:+d} fallos, coste ×{coste:.2f} y latencia ×{latencia:.2f} frente al actual"
        if int(df["error"].notna().sum()) > 0:
            decision, motivo = DESCARTADA, f"Con preguntas sin respuesta ({int(df['error'].notna().sum())}). {detalle}"
        elif d_fallos >= 2:
            decision, motivo = DESCARTADA, f"Peor calidad ({detalle})"
        elif d_fallos <= -2:
            decision, motivo = CANDIDATA, f"Mejor calidad ({detalle}); se confirmaría en el oficial y en el hold-out"
        elif coste <= 0.7:
            decision, motivo = CANDIDATA, (f"Calidad equivalente y más barato ({detalle}). Solo medido en el golden propio y en "
                                           f"una tirada: se confirmaría en el oficial y en el hold-out")
        else:
            decision, motivo = DESCARTADA, f"Sin ventaja ({detalle})"
        opcion(area, nombre, linea(df), decision, motivo)
else:
    opcion(area, "Otros modelos (gemini-3.5-flash-lite, deepseek-v4-flash)", "sin datos: la parte B no se ha ejecutado",
           SIN_EVALUAR, "Se mantiene el modelo del baseline")

In [ ]:
decisiones = pd.DataFrame(FILAS)


def a_markdown_decisiones(t):
    """La tabla en Markdown; el área solo se escribe en la primera fila de cada bloque."""
    lineas = ["| Área | Opción | Evidencia (de los CSV) | Decisión | Motivo |", "|---|---|---|---|---|"]
    anterior = None
    for _, f in t.iterrows():
        area = f"**{f['área']}**" if f["área"] != anterior else ""
        anterior = f["área"]
        lineas.append(f"| {area} | {f['opción']} | {f['evidencia (CSV)']} | {f['decisión']} | {f['motivo']} |")
    return "\n".join(lineas)


texto_decisiones = a_markdown_decisiones(decisiones)
display(Markdown(texto_decisiones))

decisiones.to_csv(RES / "tabla_decisiones.csv", index=False)
(RES / "tabla_decisiones.md").write_text(texto_decisiones, encoding="utf-8")
print(f"Guardado: resultados/tabla_decisiones.{{csv,md}}  ({len(decisiones)} opciones)")
print()
print(decisiones["decisión"].value_counts().to_string())

## Qué sistema dejar por defecto

`responder()` y `evaluar()` usan por defecto la configuración `CONFIG_POR_DEFECTO` de `agente/agente.py`, y esa es la que se ejecuta el día 24 sobre las preguntas ciegas. Hoy es `"final"`, pero eso solo es correcto si la tabla lo respalda. Si un sistema intermedio resulta mejor o más barato con la misma calidad, se cambia esa constante.

La celda siguiente cuenta, para cada sistema, cuántos **fallos** comete en total (cifra, cita y trayectoria, sobre los dos golden; menos es mejor) para ayudar a decidirlo.

In [ ]:
def total_fallos(df):
    """Fallos totales de cifra, cita y trayectoria."""
    return sum(fallos(df, columna) for columna in ("cifra_ok", "cita_ok", "tool_ok"))


print(f"Configuración por defecto hoy: {ag.CONFIG_POR_DEFECTO!r}\n")
for sistema in NOMBRES:
    total = total_fallos(EVAL[sistema, "propio"]) + total_fallos(EVAL[sistema, "oficial"])
    coste = (EVAL[sistema, "propio"]["coste_usd"].mean() + EVAL[sistema, "oficial"]["coste_usd"].mean()) / 2 * 100
    print(f"  {NOMBRES[sistema]:<20} {total:>3} fallos · {coste:.2f} ¢ por pregunta")

## Cómo leer todo esto en la presentación

- **La respuesta a "¿mejoró algo y a qué coste?"** está en la tabla y en la ablación: qué mejoras movieron la aguja, cuántas preguntas, y cuánto subió el coste y la latencia.
- **Una mejora que no movió nada es un resultado.** Se cuenta igual: "probamos X, no cambió la métrica, y esto explica por qué".
- **Modelos y embeddings.** Se midieron aparte, en el apartado de sensibilidad: el embedding y el LLM de reescritura, a nivel de retrieval, y el modelo del agente, de punta a punta si se activa. Que el mejor embedding apenas supere al control es un hallazgo, y explica por qué no se cambió.
- **Reservas que hay que decir en voz alta:**
  - Las mejoras se eligieron mirando estas mismas preguntas, así que el resultado sobre el golden propio es algo optimista. El hold-out del día 24 es la comprobación real.
  - Con 13 preguntas evaluables, las diferencias pequeñas son ruido.
  - El modelo no es determinista: la misma tirada repetida puede variar en una o dos preguntas.
- **Si el hold-out sale peor que el golden propio, no es un suspenso: es un hallazgo.** Hay que explicar qué parte de la mejora era general y cuál era memoria del conjunto con el que se iteró.

**Siguiente paso:** el notebook 06 deja preparado el lanzador del día 24, y el ensayo del clon limpio comprueba que `responder()` y `evaluar()` funcionan sin tocar nada.

In [ ]:
print("NOTEBOOK 05 COMPLETADO — sistemas comparados.")
print("Ficheros: resultados/eval_<sistema>_<golden>.csv, tabla_baseline_vs_final_propio.{csv,md} y tabla_decisiones.{csv,md}")
print("Siguiente: notebook 06 (ciegas) y el ensayo del clon limpio")